<a href="https://colab.research.google.com/github/emurlu024/UTokyo-Muon-DataAnalysis/blob/main/Run1_AX19.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Connecting to GitHub

YouTo connect your Colab notebook to GitHub, follow these steps:

1.  Go to **File** in the Colab menu bar.
2.  You have a few options:
    *   **Save a copy in GitHub**: This will save the current state of your notebook as a new file in a specified GitHub repository.
    *   **Open from GitHub**: This allows you to open an existing notebook directly from a GitHub repository.
    *   **Save a copy as a GitHub Gist**: This saves your notebook as a Gist on GitHub.

Choose the option that best suits your needs and follow the prompts to authenticate with GitHub and select your repository.

In [ ]:
#!/usr/bin/env python3
"""Compare Flag-1 SiPM amplitudes from Run 1 and AX detector 019."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


# In Colab, upload the two filtered files and change these paths if necessary.
RUN_FILE = Path("filtered_muon_data/filtered_run1.txt")
AX_FILE = Path("filtered_muon_data/filtered_AxLab_C_019.txt")
OUTPUT_FILE = Path("run1_ax019_flag1_histogram.png")


def load_flag1_sipm(path: Path) -> np.ndarray:
    """Read every non-comment row and return Flag-1 SiPM values in mV."""
    values = []
    with path.open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip() or line.lstrip().startswith("#"):
                continue

            fields = line.split()
            if len(fields) < 5:
                raise ValueError(f"Malformed data row in {path}, line {line_number}")
            if fields[2] == "1":
                values.append(float(fields[4]))

    if not values:
        raise ValueError(f"No Flag-1 events found in {path}")
    return np.asarray(values)


def main() -> None:
    run_signal = load_flag1_sipm(RUN_FILE)
    ax_signal = load_flag1_sipm(AX_FILE)

    plot_limit = 240
    bins = np.arange(0, plot_limit + 20, 20)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(
        run_signal,
        bins=bins,
        histtype="step",
        linewidth=2.3,
        color="#276FBF",
        label=f"Filtered Run 1 (N={len(run_signal)})",
    )
    ax.hist(
        ax_signal,
        bins=bins,
        histtype="step",
        linewidth=2.3,
        color="#D1495B",
        label=f"Filtered AX 019 (N={len(ax_signal)})",
    )

    run_overflow = int((run_signal >= plot_limit).sum())
    ax_overflow = int((ax_signal >= plot_limit).sum())
    if run_overflow or ax_overflow:
        ax.text(
            0.98,
            0.76,
            f"Outside displayed range (≥{plot_limit} mV):\n"
            f"Run 1 = {run_overflow}, AX 019 = {ax_overflow}",
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=10,
            bbox={"facecolor": "white", "edgecolor": "0.8", "alpha": 0.92},
        )

    ax.set_xlim(0, plot_limit)
    ax.set_xticks(np.arange(0, plot_limit + 1, 20))
    ax.set_xlabel("SiPM signal amplitude (mV)")
    ax.set_ylabel("Number of Flag-1 events")
    ax.set_title("Flag-1 SiPM distributions: Run 1 vs AX 019")
    ax.grid(alpha=0.25, linestyle="--")
    ax.legend(frameon=False)

    fig.text(
        0.5,
        0.01,
        "Shared 20 mV bins; these are independently filtered events, not only timestamp matches",
        ha="center",
        fontsize=9.5,
    )
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    fig.savefig(OUTPUT_FILE, dpi=300, bbox_inches="tight")

    print(f"Run 1: N={len(run_signal)}, median={np.median(run_signal):.1f} mV")
    print(f"AX 019: N={len(ax_signal)}, median={np.median(ax_signal):.1f} mV")
    print(f"Saved {OUTPUT_FILE.resolve()}")


if __name__ == "__main__":
    main()

FileNotFoundError: [Errno 2] No such file or directory: 'filtered_muon_data/filtered_run1.txt'